# IndexTTS2 - 部署笔记本 (Colab/Kaggle)

**mamba环境**: 使用mamba创建独立Python 3.10环境，所有后续操作都在此环境中进行

In [ ]:
# ============================================
# Cell 1: 安装mamba + 创建Python 3.10环境
# ============================================
import os, sys, json, subprocess, urllib.request

# 环境检测
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
WORK_DIR = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/tmp'
REPO_DIR = f"{WORK_DIR}/index-tts"
MAMBA_ROOT = f"{WORK_DIR}/mamba"
MAMBA_ENV = f"{MAMBA_ROOT}/envs/tts"
MICROMAMBA_BIN = None  # 将在install_micromamba中设置

print(f"🔍 环境: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")
print(f"📁 工作目录: {WORK_DIR}")

# ===== 工具函数 =====
def shell(cmd, check=True):
    """执行shell命令"""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f"❌ 命令失败: {cmd}")
        print(f"错误: {result.stderr}")
        return False, result.stderr
    return True, result.stdout

def check_mamba_cmd():
    """检查mamba命令是否可用"""
    global MICROMAMBA_BIN
    try:
        # 优先使用已知的micromamba路径
        cmd = MICROMAMBA_BIN if MICROMAMBA_BIN else "mamba"
        result = subprocess.run([cmd, "--version"], capture_output=True, text=True)
        if result.returncode != 0:
            return False
        output = result.stdout.lower() + result.stderr.lower()
        return 'mamba' in output and 'coverage' not in output
    except:
        return False

def install_micromamba():
    """安装micromamba - 强制安装并确保可用"""
    print("🔧 强制安装 micromamba...")
    try:
        # 设置mamba根目录
        os.environ['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT
        
        # 下载安装脚本
        installer_url = "https://micro.mamba.pm/install.sh"
        install_script = f"{WORK_DIR}/install.sh"
        urllib.request.urlretrieve(installer_url, install_script)
        os.chmod(install_script, 0o755)
        
        # 执行安装
        result = subprocess.run(['bash', install_script], capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        
        # 查找micromamba安装位置
        possible_paths = [
            "/root/.local/bin/micromamba",
            "/usr/local/bin/micromamba",
            f"{os.environ.get('HOME', '/root')}/.local/bin/micromamba",
            f"{MAMBA_ROOT}/bin/micromamba",
        ]
        
        MICROMAMBA_BIN = None
        for path in possible_paths:
            if os.path.exists(path):
                MICROMAMBA_BIN = path
                print(f"✅ 找到micromamba: {path}")
                break
        
        if not MICROMAMBA_BIN:
            # 尝试在PATH中查找
            result = subprocess.run(['which', 'micromamba'], capture_output=True, text=True)
            if result.returncode == 0:
                MICROMAMBA_BIN = result.stdout.strip()
                print(f"✅ 在PATH中找到micromamba: {MICROMAMBA_BIN}")
            else:
                print("❌ 找不到micromamba二进制文件")
                return False
        
        # 添加到PATH
        bin_dir = os.path.dirname(MICROMAMBA_BIN)
        os.environ['PATH'] = f"{bin_dir}:{os.environ['PATH']}"
        
        # 创建mamba别名
        mamba_link = os.path.join(bin_dir, "mamba")
        if not os.path.exists(mamba_link):
            os.symlink(MICROMAMBA_BIN, mamba_link)
            print(f"✅ 创建mamba链接: {mamba_link}")
        
        # 验证安装并设置全局变量
        result = subprocess.run([MICROMAMBA_BIN, "--version"], capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ micromamba验证成功: {result.stdout.strip()}")
            # 设置全局变量供其他函数使用
            globals()['MICROMAMBA_BIN'] = MICROMAMBA_BIN
            return True
        else:
            print("❌ micromamba验证失败")
            return False
            
    except Exception as e:
        print(f"❌ micromamba安装失败: {e}")
        import traceback
        traceback.print_exc()
        return False

def create_mamba_env():
    """创建mamba环境 (Python 3.10 + CUDA)"""
    global MICROMAMBA_BIN
    print("\n🚀 创建 mamba 环境 (Python 3.10 + CUDA 11.8)...")
    print("⏳ 这可能需要几分钟...")
    
    # 使用实际的micromamba路径
    cmd_bin = MICROMAMBA_BIN if MICROMAMBA_BIN else "mamba"
    cmd = f"{cmd_bin} create -y -p {MAMBA_ENV} python=3.10 cudatoolkit=11.8 -c conda-forge -c nvidia"
    success, output = shell(cmd)
    
    if success:
        print("✅ mamba环境创建完成")
        return True
    else:
        print("⚠️ 环境创建失败")
        return False

def get_mamba_python():
    """获取mamba环境的python路径"""
    return f"{MAMBA_ENV}/bin/python"

def get_mamba_pip():
    """获取mamba环境的pip路径"""
    return f"{MAMBA_ENV}/bin/pip"

# ===================================================

# 步骤1: 安装mamba
print("\n📦 步骤1: 安装mamba...")
if not check_mamba_cmd():
    install_micromamba()
else:
    print("✅ mamba已可用")

# 步骤2: 创建mamba环境
print("\n📦 步骤2: 创建环境...")
ENV_CREATED = False
if check_mamba_cmd():
    ENV_CREATED = create_mamba_env()
else:
    print("⚠️ mamba不可用，将使用系统Python")

# 步骤3: 安装PyTorch (使用mamba环境的pip)
print("\n📦 步骤3: 安装PyTorch...")
if ENV_CREATED:
    pip_cmd = get_mamba_pip()
    python_cmd = get_mamba_python()
else:
    pip_cmd = "pip"
    python_cmd = "python"

shell(f"{pip_cmd} install -q torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu121")
print("✅ PyTorch安装完成")

# 步骤4: 安装基础依赖
print("\n📦 步骤4: 安装基础依赖...")
basic_deps = ["pyngrok", "gradio==5.45.0", "fastapi", "uvicorn", "python-multipart", "huggingface-hub"]
for dep in basic_deps:
    shell(f"{pip_cmd} install -q {dep}")
print("✅ 基础依赖安装完成")

# 步骤5: 克隆代码
print("\n📦 步骤5: 克隆代码...")
if os.path.exists(REPO_DIR):
    shell(f"rm -rf {REPO_DIR}")
shell(f"git clone -b dev https://github.com/infinite-gaming-studio/index-tts.git {REPO_DIR}")
print("✅ 代码克隆完成")

# 保存配置
config = {
    "work_dir": WORK_DIR,
    "repo_dir": REPO_DIR,
    "mamba_env": MAMBA_ENV if ENV_CREATED else None,
    "mamba_python": get_mamba_python() if ENV_CREATED else "python",
    "mamba_pip": get_mamba_pip() if ENV_CREATED else "pip",
    "in_colab": IN_COLAB,
    "in_kaggle": IN_KAGGLE,
    "env_created": ENV_CREATED
}

with open("/tmp/notebook_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\n{'='*60}")
print("✅ Cell 1 完成!")
print(f"   环境: {'✅ mamba' if ENV_CREATED else '⚠️ 系统Python'}")
if ENV_CREATED:
    print(f"   Python: {get_mamba_python()}")
    print(f"   pip: {get_mamba_pip()}")
print(f"   代码: {REPO_DIR}")
print(f"\n👉 请停止当前kernel，然后选择mamba环境的Python再运行Cell 2")
print(f"   (在Colab中: 运行时 -> 更改运行时类型 -> 选择本地mamba路径)")
print(f"{'='*60}")



In [ ]:
# ============================================
# Cell 2: 完整安装 + 模型下载
# 使用mamba环境的Python运行此Cell
# ============================================
import sys, os, json, subprocess

# 加载配置
with open("/tmp/notebook_config.json", "r") as f:
    config = json.load(f)

REPO_DIR = config["repo_dir"]
MAMBA_ENV = config.get("mamba_env")
ENV_CREATED = config.get("env_created", False)

# 设置命令前缀
if ENV_CREATED and MAMBA_ENV and os.path.exists(MAMBA_ENV):
    PYTHON_CMD = config["mamba_python"]
    PIP_CMD = config["mamba_pip"]
    print(f"✅ 使用mamba环境: {MAMBA_ENV}")
else:
    PYTHON_CMD = "python"
    PIP_CMD = "pip"
    print("⚠️ 使用系统Python")

print(f"   Python: {PYTHON_CMD}")
print(f"   pip: {PIP_CMD}")

# 添加项目路径
sys.path.insert(0, REPO_DIR)

# 使用mamba的python运行后续脚本
print("\n📦 安装项目依赖...")
result = subprocess.run([
    PYTHON_CMD, "-c",
    f"""
import sys
sys.path.insert(0, '{REPO_DIR}')
from deploy.utils import DependencyInstaller, ModelDownloader, check_model_exists
DependencyInstaller.install_deps(use_mamba=False)
"""
])

if result.returncode != 0:
    print("⚠️ 依赖安装可能有问题，继续执行...")

# 安装项目本身
print("\n📦 安装项目...")
subprocess.run([PIP_CMD, "install", "-q", "-e", REPO_DIR], check=True)

# 下载模型
CHECKPOINT_DIR = f"{REPO_DIR}/checkpoints"
print("\n🤖 检查模型...")

# 使用mamba python检查并下载模型
model_check = subprocess.run([
    PYTHON_CMD, "-c",
    f"""
import sys
sys.path.insert(0, '{REPO_DIR}')
from deploy.utils import check_model_exists, ModelDownloader
import os
checkpoint_dir = '{CHECKPOINT_DIR}'
if not check_model_exists(checkpoint_dir):
    print('Downloading model...')
    ModelDownloader.download(checkpoint_dir, source='huggingface')
else:
    print('Model already exists')
"""
], capture_output=True, text=True)

print(model_check.stdout)
if model_check.stderr:
    print(model_check.stderr)

print(f"\n{'='*60}")
print("✅ Cell 2 完成! 可以启动服务了")
print(f"{'='*60}")

In [ ]:
# ============================================
# Cell 3: 启动服务
# 使用mamba环境的Python启动服务
# ============================================
import json, subprocess, sys, os

# 加载配置
with open("/tmp/notebook_config.json", "r") as f:
    config = json.load(f)

REPO_DIR = config["repo_dir"]
PYTHON_CMD = config.get("mamba_python", "python")
ENV_CREATED = config.get("env_created", False)

print(f"🚀 使用 {'mamba' if ENV_CREATED else '系统'} Python启动服务...")
print(f"   Python: {PYTHON_CMD}")

PORT = 8000
MODE = "both"
NGROK_TOKEN = None  # 替换为你的token

# 使用mamba python启动服务
launch_script = f"""
import sys
sys.path.insert(0, '{REPO_DIR}')
from deploy.launcher import quick_start
launcher = quick_start(port={PORT}, mode='{MODE}', ngrok_token={NGROK_TOKEN})
import time
time.sleep(2)
print('Service started!')
"""

# 后台启动服务
import time
print("⏳ 启动服务中...")
process = subprocess.Popen(
    [PYTHON_CMD, "-c", launch_script],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)
print(f"\n✅ 服务启动命令已执行")
print(f"   PID: {process.pid}")
print(f"   查看日志请运行Cell 4")

In [ ]:
# ============================================
# Cell 4: 服务管理
# ============================================
import subprocess

# 查看服务状态
print("📊 检查服务状态...")
!ps aux | grep -E "(python|service)" | grep -v grep | head -5

# 查看日志（如果有日志文件的话）
log_file = f"{REPO_DIR}/service.log"
if os.path.exists(log_file):
    print(f"\n📝 最新日志 ({log_file}):")
    !tail -n 20 {log_file}
else:
    print("\nℹ️ 日志文件不存在")